In [ ]:
%pip install requests pyarrow numpy pandas plotly scipy google-cloud-storage

In [ ]:
import os
import sys

REPO_URL = "https://github.com/payamdav/pycrypto.git"
REPO_NAME = "pycrypto"
BRANCH_NAME = "claude/rsi-filtering-label-histograms-ywap79"

if not os.path.exists(REPO_NAME):
    os.system(f'git clone -b {BRANCH_NAME} {REPO_URL}')

REPO_PATH = os.path.abspath(REPO_NAME)
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

In [ ]:
import io
import warnings
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde

warnings.filterwarnings('ignore')

In [ ]:
sys.path.insert(0, os.path.join(REPO_PATH, 'packages', 'tools', 'google_cloud_storage_tools'))
from gcs_tools import gcs_json_key_file, read_file

gcs_json_key_file()

In [ ]:
ASSETS = ["btcusdt", "ethusdt", "trumpusdt", "vineusdt", "adausdt", "xrpusdt", "dogeusdt"]

# fl_data column indices
RSI_INDICES  = [49, 50, 51]
RSI_NAMES    = ["RSI_p7", "RSI_p14", "RSI_p60"]
LABEL_INDICES   = [52, 53, 54, 55]
LABEL_HORIZONS  = ["1h", "2h", "3h", "4h"]

# Directional filter thresholds: keep samples with rsi >= threshold
# Ridge 0 = All (no filter), then progressively exclude lower-RSI samples
FILTER_THRESHOLDS = [round(-0.9 + 0.1 * i, 1) for i in range(19)]  # -0.9 .. 0.9

BUCKET   = "payamdprojectbucket"
KDE_BW   = 0.05
MAX_KDE_SAMPLES = 100_000  # cap for KDE speed

In [ ]:
def _to_rgba(color_str: str, alpha: float = 0.5) -> str:
    """Convert 'rgb(r, g, b)' or '#rrggbb' to 'rgba(r, g, b, alpha)'."""
    if color_str.startswith('#'):
        r = int(color_str[1:3], 16)
        g = int(color_str[3:5], 16)
        b = int(color_str[5:7], 16)
        return f'rgba({r},{g},{b},{alpha})'
    if color_str.startswith('rgb('):
        inner = color_str[4:-1]
        return f'rgba({inner},{alpha})'
    return color_str


def _density(data: np.ndarray, x_eval: np.ndarray, bw: float, max_n: int) -> np.ndarray:
    """Estimate KDE density on x_eval; returns zeros if too few points."""
    if len(data) < 50:
        return np.zeros_like(x_eval)
    if len(data) > max_n:
        rng = np.random.default_rng(42)
        data = rng.choice(data, size=max_n, replace=False)
    try:
        return gaussian_kde(data, bw_method=bw)(x_eval)
    except Exception:
        return np.zeros_like(x_eval)


def make_ridgeline_figure(
    fl: np.ndarray,
    rsi_idx: int,
    rsi_name: str,
    asset_name: str,
    filter_thresholds: list,
) -> go.Figure:
    """
    Directional ridgeline plot.
    Each ridge keeps samples where rsi >= threshold.
    Ridge 0 (bottom) = all samples, Ridge N (top) = rsi >= filter_thresholds[-1].
    """
    rsi = fl[:, rsi_idx]
    n_ridges = 1 + len(filter_thresholds)

    # Build boolean masks for each ridge
    masks = [np.ones(len(rsi), dtype=bool)]  # Ridge 0: All
    for t in filter_thresholds:
        masks.append(rsi >= t)

    ridge_labels = ['All'] + [f'RSI\u2265{t:.1f}' for t in filter_thresholds]
    n_samples    = [int(m.sum()) for m in masks]

    # Color: dark (all) -> bright (most filtered)
    palette = px.colors.sample_colorscale('plasma', np.linspace(0.0, 0.92, n_ridges).tolist())

    x_eval = np.linspace(-1.02, 1.02, 300)

    fig = make_subplots(
        rows=1, cols=4,
        subplot_titles=LABEL_HORIZONS,
        horizontal_spacing=0.04,
    )

    for col_idx in range(4):
        label_col = fl[:, LABEL_INDICES[col_idx]]

        # KDE density per ridge
        densities = [_density(label_col[m], x_eval, KDE_BW, MAX_KDE_SAMPLES) for m in masks]

        # Spacing so ridges overlap nicely
        max_d = max((d.max() for d in densities if d.max() > 0), default=1.0)
        spacing = max_d * 1.15

        for ridge_idx in range(n_ridges):
            density = densities[ridge_idx]
            offset  = ridge_idx * spacing
            color   = palette[ridge_idx]
            fill_c  = _to_rgba(color, 0.55)

            # Closed polygon: curve top then flat baseline back
            x_poly = np.concatenate([x_eval, x_eval[::-1]]).tolist()
            y_poly = np.concatenate([density + offset, np.full(len(x_eval), offset)]).tolist()

            fig.add_trace(
                go.Scatter(
                    x=x_poly,
                    y=y_poly,
                    fill='toself',
                    fillcolor=fill_c,
                    line=dict(color=color, width=0.8),
                    mode='lines',
                    name=f"{ridge_labels[ridge_idx]} (n={n_samples[ridge_idx]:,})",
                    legendgroup=ridge_labels[ridge_idx],
                    showlegend=(col_idx == 0),
                    hovertemplate=(
                        f"{ridge_labels[ridge_idx]}<br>"
                        f"n={n_samples[ridge_idx]:,}<br>"
                        "label=%{x:.3f}<extra></extra>"
                    ),
                ),
                row=1, col=col_idx + 1,
            )

        # Y-axis: show ridge labels on first subplot, hide on others
        if col_idx == 0:
            tick_vals = [i * spacing + max_d * 0.18 for i in range(n_ridges)]
            fig.update_yaxes(
                tickvals=tick_vals,
                ticktext=ridge_labels,
                tickfont=dict(size=7),
                showgrid=False,
                row=1, col=1,
            )
        else:
            fig.update_yaxes(visible=False, row=1, col=col_idx + 1)

    fig.update_xaxes(range=[-1.05, 1.05], title_text='Normalized Label')
    fig.update_layout(
        title=dict(
            text=f"{asset_name.upper()} — VWAP Label Histograms | Directional RSI Filter ({rsi_name})",
            font=dict(size=14),
        ),
        height=180 + n_ridges * 32,
        width=1400,
        plot_bgcolor='white',
        paper_bgcolor='white',
        legend=dict(
            orientation='v',
            x=1.01, y=1.0,
            font=dict(size=8),
            traceorder='normal',
        ),
        margin=dict(r=200, t=80, b=50, l=120),
    )
    return fig

In [ ]:
# ── Main loop ──────────────────────────────────────────────────────────────
# For each asset: load fl_data, then produce 3 ridgeline figures
# (one per RSI period), each with 4 subplots (one per label horizon).
# Each figure shows 20 ridges: All + RSI >= [-0.9 .. 0.9 step 0.1].

for asset in ASSETS:
    print(f"\n{'='*60}")
    print(f"  {asset.upper()}")
    print(f"{'='*60}")

    try:
        data_bytes = read_file(BUCKET, f"fl_data_{asset}")
        fl = np.load(io.BytesIO(data_bytes))
        print(f"  Loaded {fl.shape[0]:,} observations  shape={fl.shape}")
    except Exception as e:
        print(f"  SKIP — could not load fl_data_{asset}: {e}")
        continue

    for rsi_idx, rsi_name in zip(RSI_INDICES, RSI_NAMES):
        print(f"  Plotting {rsi_name} ...")
        fig = make_ridgeline_figure(fl, rsi_idx, rsi_name, asset, FILTER_THRESHOLDS)
        fig.show()

    del fl  # free memory before next asset

print("\nDone.")